# Model Training

Pada notebook ini, dilakukan pelatihan satu model baseline untuk keperluan eksplorasi dan pemahaman awal terhadap data.

Model yang digunakan hanya 1 saja untuk uji coba, model lainnya akan di gunakan pada pipeline, termasuk proses hyperparameter tuning otomatis dan resampling, dijalankan di dalam training pipeline (pendekatan AutoML).

Model pada tahap ini dioptimalkan untuk recall, dengan tujuan meminimalkan jumlah nasabah yang berpotensi default namun tidak terdeteksi (false negative).

Teknik resampling diterapkan di dalam pipeline untuk menghindari terjadinya data leakage selama proses training.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
import os
sys.path.append(os.path.abspath(".."))

In [2]:
from src.utils.utils import ConfigManager
import pandas as pd 
import copy
import json
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE 
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline
import optuna
from sklearn.model_selection import cross_val_score
import time
from datetime import datetime



# Load Config

In [3]:
# Usage configuration manager
config_manager = ConfigManager('config/config.yaml')
# Load the config
config = config_manager.load_config()
config

{'features': {'cat_nominal': ['person_home_ownership', 'loan_intent'],
  'cat_ordinal': {'cb_person_default_on_file': {'N': 0, 'Y': 1},
   'loan_grade': {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}},
  'categorical': ['person_home_ownership',
   'loan_intent',
   'loan_grade',
   'cb_person_default_on_file'],
  'numerical': ['person_age',
   'person_income',
   'person_emp_length',
   'loan_amnt',
   'loan_int_rate',
   'loan_percent_income',
   'cb_person_cred_hist_length']},
 'path': {'interim_data': '/Users/apa/Documents/DATAKU/BootCamp/Pacmann/ML_API/mentoring/new_credit_scoring/data/interim',
  'logrob_preprocessing': '/Users/apa/Documents/DATAKU/BootCamp/Pacmann/ML_API/mentoring/new_credit_scoring/models/logrob_prep_pipeline.pkl',
  'models': '/Users/apa/Documents/DATAKU/BootCamp/Pacmann/ML_API/mentoring/new_credit_scoring/models',
  'experiments': '/Users/apa/Documents/DATAKU/BootCamp/Pacmann/ML_API/mentoring/new_credit_scoring/models/experiments/training_log.json',


In [4]:
# Data path
train_data_path = config['path']['train_data_processed_pipe']
# Deserialization
data_train = config_manager.deserialize_data(train_data_path)

In [5]:
data_train

{'Robust Scaler': {'X_train':        num__person_age  num__person_income  num__person_emp_length  \
  352          -0.428571           -0.959764                    -0.2   
  12819        -0.428571            0.738280                     0.6   
  19720         1.000000           -0.483327                     0.6   
  27174         0.428571            1.722653                     1.6   
  8274         -0.285714            0.049219                    -0.8   
  ...                ...                 ...                     ...   
  29808         2.428571           -0.393749                     1.6   
  5396         -0.285714           -0.246093                     0.8   
  866          -0.714286           -0.787498                     0.4   
  15801        -0.285714            0.595546                    -0.2   
  23660         0.285714            0.246093                    -0.2   
  
         num__loan_amnt  num__loan_int_rate  num__loan_percent_income  \
  352         -0.795139         

# Training Models

In [6]:
def _update_training_log(current_log, path_log):
    current_log = copy.deepcopy(current_log)

    # If training log exist
    try:
        with open(path_log, 'r') as file:
            last_log = json.load(file)
    # if training log not exist
    except FileNotFoundError as err:
        with open(path_log, 'w') as file:
            file.write('[]')
        file.close()

        # Reload training log that we knows as last log
        with open(path_log, 'r') as file:
            last_log = json.load(file)

    # Add the current log to last log
    last_log.append(current_log)

    # Rewrite the training log with the updated one
    with open(path_log, 'w') as file:
        json.dump(last_log, file, indent=4)
        file.close()

    return last_log

    

In [7]:
def get_model_and_params(trial, model_name):

    if model_name == 'RandomForestClassifier':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 200),
            'max_depth': trial.suggest_int('max_depth', 2, 40),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
            'random_state' : 42
        }

    else:
        raise ValueError(f"Model {model_name} not supported")

    return params

def create_model_object():
    rfc = RandomForestClassifier

    list_of_models = [
        {'model_name': 'RandomForestClassifier', 'model_object': rfc}
    ]
    return list_of_models

In [8]:
# Function to create objective
def create_objective(X_train, y_train, resampler, model_name, model_obj):

    def objective(trial):
        params = get_model_and_params(trial, model_name)

        model = model_obj(**params)

        pipeline = Pipeline([
            ('resampler', resampler),
            ('model', model)
        ])

        scores = cross_val_score(
            pipeline,
            X_train,
            y_train,
            cv=5,
            scoring='recall',
            n_jobs=-1
        )

        return scores.mean()

    return objective

In [9]:
def running_train_model(data: dict):
    resamplers = {
        'SMOTE' : SMOTE(random_state=42),
        'SMOTEENN' : SMOTEENN(random_state=42)
    }
    # Get list of models
    models = create_model_object()

    # Experiment Log
    experiments_log = list()

    for prep_name, data_train in  data.items():
        X_train = data_train['X_train']
        y_train = data_train['y_train']
        for res_name, resampler in resamplers.items():
            for model in models:
                model_name = model['model_name']
                model_obj = model['model_object']
                print(f"Training: {prep_name} | {res_name} | {model_name}")
                # Create objective function for optuna
                objective = create_objective(
                    X_train,
                    y_train,
                    resampler,
                    model_name,
                    model_obj
                )
                # Optuna
                study = optuna.create_study(direction="maximize")
                start_time = time.time()
                study.optimize(objective, n_trials=5)
                finish_time = time.time()

                # 
                training_time = finish_time - start_time
                
                # Training Log Result
                log_entry = {
                    'timestamp' : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                    'preprocessing' : prep_name,
                    'resampler' : res_name,
                    'model' : model_name,
                    'recall_scores' : study.best_value,
                    'params' : study.best_params,
                    'training_time' : training_time
                }
                print(f"[RESULT] {prep_name} | {res_name} | {model_name}Score: {study.best_value:.4f}")
                
                experiments_log.append(log_entry)

    return experiments_log


In [10]:
# Run Training
train_log = running_train_model(data_train)

[I 2026-03-24 02:27:57,181] A new study created in memory with name: no-name-59d587ce-f11d-4871-888b-612f873709ab


Training: Robust Scaler | SMOTE | RandomForestClassifier


[I 2026-03-24 02:28:01,524] Trial 0 finished with value: 0.7274840678574634 and parameters: {'n_estimators': 118, 'max_depth': 21, 'min_samples_split': 6, 'min_samples_leaf': 6}. Best is trial 0 with value: 0.7274840678574634.
[I 2026-03-24 02:28:06,770] Trial 1 finished with value: 0.7312158475519036 and parameters: {'n_estimators': 194, 'max_depth': 38, 'min_samples_split': 5, 'min_samples_leaf': 10}. Best is trial 1 with value: 0.7312158475519036.
[I 2026-03-24 02:28:11,103] Trial 2 finished with value: 0.7337823980764471 and parameters: {'n_estimators': 133, 'max_depth': 30, 'min_samples_split': 2, 'min_samples_leaf': 16}. Best is trial 2 with value: 0.7337823980764471.
[I 2026-03-24 02:28:13,306] Trial 3 finished with value: 0.7349492592199709 and parameters: {'n_estimators': 100, 'max_depth': 32, 'min_samples_split': 18, 'min_samples_leaf': 17}. Best is trial 3 with value: 0.7349492592199709.
[I 2026-03-24 02:28:17,617] Trial 4 finished with value: 0.7377489099776148 and paramete

[RESULT] Robust Scaler | SMOTE | RandomForestClassifierScore: 0.7377
Training: Robust Scaler | SMOTEENN | RandomForestClassifier


[I 2026-03-24 02:28:21,117] Trial 0 finished with value: 0.7965445678397838 and parameters: {'n_estimators': 105, 'max_depth': 38, 'min_samples_split': 20, 'min_samples_leaf': 18}. Best is trial 0 with value: 0.7965445678397838.
[I 2026-03-24 02:28:23,989] Trial 1 finished with value: 0.7897784051809722 and parameters: {'n_estimators': 75, 'max_depth': 33, 'min_samples_split': 12, 'min_samples_leaf': 11}. Best is trial 0 with value: 0.7965445678397838.
[I 2026-03-24 02:28:28,524] Trial 2 finished with value: 0.7853459648092087 and parameters: {'n_estimators': 174, 'max_depth': 38, 'min_samples_split': 13, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.7965445678397838.
[I 2026-03-24 02:28:31,035] Trial 3 finished with value: 0.7944458497550679 and parameters: {'n_estimators': 75, 'max_depth': 7, 'min_samples_split': 19, 'min_samples_leaf': 18}. Best is trial 0 with value: 0.7965445678397838.
[I 2026-03-24 02:28:33,815] Trial 4 finished with value: 0.8117115867407584 and paramete

[RESULT] Robust Scaler | SMOTEENN | RandomForestClassifierScore: 0.8117
Training: Log + Robust Scaler | SMOTE | RandomForestClassifier


[I 2026-03-24 02:28:35,171] Trial 0 finished with value: 0.7365823208296954 and parameters: {'n_estimators': 62, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 16}. Best is trial 0 with value: 0.7365823208296954.
[I 2026-03-24 02:28:36,992] Trial 1 finished with value: 0.7386832148792475 and parameters: {'n_estimators': 93, 'max_depth': 11, 'min_samples_split': 17, 'min_samples_leaf': 20}. Best is trial 1 with value: 0.7386832148792475.
[I 2026-03-24 02:28:40,895] Trial 2 finished with value: 0.7377486379820102 and parameters: {'n_estimators': 191, 'max_depth': 19, 'min_samples_split': 10, 'min_samples_leaf': 12}. Best is trial 1 with value: 0.7386832148792475.
[I 2026-03-24 02:28:43,077] Trial 3 finished with value: 0.7337834860588652 and parameters: {'n_estimators': 98, 'max_depth': 27, 'min_samples_split': 16, 'min_samples_leaf': 7}. Best is trial 1 with value: 0.7386832148792475.
[I 2026-03-24 02:28:47,406] Trial 4 finished with value: 0.7316823200137086 and parameter

[RESULT] Log + Robust Scaler | SMOTE | RandomForestClassifierScore: 0.7387
Training: Log + Robust Scaler | SMOTEENN | RandomForestClassifier


[I 2026-03-24 02:28:51,234] Trial 0 finished with value: 0.7832491506937249 and parameters: {'n_estimators': 81, 'max_depth': 9, 'min_samples_split': 4, 'min_samples_leaf': 1}. Best is trial 0 with value: 0.7832491506937249.
[I 2026-03-24 02:28:54,849] Trial 1 finished with value: 0.7853467807960224 and parameters: {'n_estimators': 189, 'max_depth': 5, 'min_samples_split': 20, 'min_samples_leaf': 5}. Best is trial 1 with value: 0.7853467807960224.
[I 2026-03-24 02:28:58,455] Trial 2 finished with value: 0.775313950926553 and parameters: {'n_estimators': 110, 'max_depth': 35, 'min_samples_split': 2, 'min_samples_leaf': 1}. Best is trial 1 with value: 0.7853467807960224.
[I 2026-03-24 02:29:01,755] Trial 3 finished with value: 0.7855787930467043 and parameters: {'n_estimators': 101, 'max_depth': 38, 'min_samples_split': 20, 'min_samples_leaf': 8}. Best is trial 3 with value: 0.7855787930467043.
[I 2026-03-24 02:29:04,819] Trial 4 finished with value: 0.7944463937462769 and parameters: {'

[RESULT] Log + Robust Scaler | SMOTEENN | RandomForestClassifierScore: 0.7944


In [11]:
train_log

[{'timestamp': '2026-03-24 02:28:17',
  'preprocessing': 'Robust Scaler',
  'resampler': 'SMOTE',
  'model': 'RandomForestClassifier',
  'recall_scores': 0.7377489099776148,
  'params': {'n_estimators': 189,
   'max_depth': 32,
   'min_samples_split': 19,
   'min_samples_leaf': 19},
  'training_time': 20.437237977981567},
 {'timestamp': '2026-03-24 02:28:33',
  'preprocessing': 'Robust Scaler',
  'resampler': 'SMOTEENN',
  'model': 'RandomForestClassifier',
  'recall_scores': 0.8117115867407584,
  'params': {'n_estimators': 187,
   'max_depth': 3,
   'min_samples_split': 8,
   'min_samples_leaf': 7},
  'training_time': 16.19541096687317},
 {'timestamp': '2026-03-24 02:28:47',
  'preprocessing': 'Log + Robust Scaler',
  'resampler': 'SMOTE',
  'model': 'RandomForestClassifier',
  'recall_scores': 0.7386832148792475,
  'params': {'n_estimators': 93,
   'max_depth': 11,
   'min_samples_split': 17,
   'min_samples_leaf': 20},
  'training_time': 13.589770078659058},
 {'timestamp': '2026-03-

In [12]:
PATH_TRAINING_LOG = config['path']['experiments']
print(PATH_TRAINING_LOG)
last_log = _update_training_log(train_log, PATH_TRAINING_LOG)

/Users/apa/Documents/DATAKU/BootCamp/Pacmann/ML_API/mentoring/new_credit_scoring/models/experiments/training_log.json


In [13]:
# view with pandas
def _load_best_experiment(training_log: list, metrics: str='recall_scores'):
    flat_logs = list()

    for sublist in training_log:
        for item in sublist:
            flat_logs.append(item)

    df = pd.DataFrame(flat_logs)
    # sort get the best index
    df = df.sort_values(by=metrics, ascending=False).reset_index(drop=True)
    best = df.iloc[0]
    return best, df

In [14]:
best_result, df = _load_best_experiment(training_log=last_log)
df

,timestamp,preprocessing,resampler,model,recall_scores,params,training_time
0,2026-03-24 02:28:33,Robust Scaler,SMOTEENN,RandomForestClassifier,0.811712,"{'n_estimators': 187, 'max_depth': 3, 'min_sam...",16.195411
1,2026-03-24 02:29:04,Log + Robust Scaler,SMOTEENN,RandomForestClassifier,0.794446,"{'n_estimators': 189, 'max_depth': 4, 'min_sam...",17.412253
2,2026-03-24 02:28:47,Log + Robust Scaler,SMOTE,RandomForestClassifier,0.738683,"{'n_estimators': 93, 'max_depth': 11, 'min_sam...",13.589770
3,2026-03-24 02:28:17,Robust Scaler,SMOTE,RandomForestClassifier,0.737749,"{'n_estimators': 189, 'max_depth': 32, 'min_sa...",20.437238


In [15]:
best_result

timestamp                                      2026-03-24 02:28:33
preprocessing                                        Robust Scaler
resampler                                                 SMOTEENN
model                                       RandomForestClassifier
recall_scores                                             0.811712
params           {'n_estimators': 187, 'max_depth': 3, 'min_sam...
training_time                                            16.195411
Name: 0, dtype: object

In [16]:
def rebuild_pipeline_from_log(best_row: any) -> Pipeline:
    # Get the info
    model_name = best_row['model']
    resampler_name = best_row['resampler']
    params = best_row['params']
    prep_name = best_row['preprocessing']

    # -- Model
    if model_name == 'RandomForestClassifier':
        model = RandomForestClassifier(**params)
    else:
        raise ValueError(f'Model {model_name} is not supported')

    # -- Resampler
    if resampler_name == 'SMOTE':
        resampler = SMOTE(random_state=42)
    elif resampler_name == 'SMOTEENN':
        resampler = SMOTEENN(random_state=42)
    else:
        raise ValueError(f"Resampler {resampler_name} is not supported")
    
    # Pipeline
    pipeline = Pipeline([
        ('resampler', resampler),
        ('model', model)
    ])

    return pipeline, prep_name

def fit_best_model(pipeline: Pipeline, prep_name: str, data_train: dict):
    X_train = data_train[prep_name]['X_train']
    y_train = data_train[prep_name]['y_train']

    pipeline.fit(X_train, y_train)
    return pipeline

In [17]:
# GEt best pipeline
best_pipeline, prep_name = rebuild_pipeline_from_log(best_row=best_result)
# Fit the best pipeline
best_model = fit_best_model(pipeline=best_pipeline, prep_name=prep_name, data_train=data_train)

In [18]:
# Save best model
PATH_MODEL = config['path']['models']
model_path = f"{PATH_MODEL}/best_model.pkl"

config_manager.serialized_data(best_model, model_path)
config_manager.update_config('path.best_model', model_path)

In [19]:
# update config for best preprocessing
config_manager.update_config('best_preprocessing', best_result['preprocessing'])